Cell 1- Create Count Vectorizer

In [1]:
# Cell 1: Load data + imports
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import warnings
warnings.filterwarnings('ignore')

# Load the cleaned data from notebook 02
train_df = pd.read_csv('train_clean.csv')
test_df = pd.read_csv('test_clean.csv')

print("Loaded:", train_df.shape, test_df.shape)
print("Columns:", train_df.columns.tolist())

Loaded: (120000, 3) (7600, 3)
Columns: ['text', 'label', 'clean_text']


Cell 2- Vectorization

In [2]:
print("Creating Count Vectorizer for LDA...")

count_vectorizer = CountVectorizer(
    max_features=5000,
    min_df=5,
    stop_words="english"
)

X_counts = count_vectorizer.fit_transform(train_df["clean_text"])

print("Shape:", X_counts.shape)

Creating Count Vectorizer for LDA...
Shape: (120000, 5000)


Cell 3- Train LDA Model

In [3]:
from sklearn.decomposition import LatentDirichletAllocation

print("Training LDA model...")

lda_model = LatentDirichletAllocation(
    n_components=4,   # we expect 4 topics (World, Sports, Business, Sci/Tech)
    random_state=42,
    learning_method="batch"
)

lda_model.fit(X_counts)

print("LDA training complete.")

Training LDA model...
LDA training complete.


Cell 4 — Extract Topics

In [4]:
def print_topics(model, vectorizer, top_n=10):
    feature_names = vectorizer.get_feature_names_out()
    
    for topic_idx, topic in enumerate(model.components_):
        print(f"\nTopic {topic_idx + 1}:")
        top_words = [feature_names[i] for i in topic.argsort()[-top_n:]]
        print(", ".join(top_words))

print_topics(lda_model, count_vectorizer)


Topic 1:
technology, said, corp, computer, internet, software, service, microsoft, company, new

Topic 2:
million, percent, company, year, price, new, stock, oil, said, reuters

Topic 3:
new, president, official, killed, people, minister, reuters, iraq, quot, said

Topic 4:
victory, night, election, bush, year, team, new, season, win, game


Cell 5 — Assign Topics to Documents

In [5]:
doc_topics = lda_model.transform(X_counts)

train_df["topic"] = doc_topics.argmax(axis=1)

train_df[["text", "label", "topic"]].head()

,text,label,topic
0,Wall St. Bears Claw Back Into the Black (Reute...,2,0
1,Carlyle Looks Toward Commercial Aerospace (Reu...,2,1
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,2,1
3,Iraq Halts Oil Exports from Main Southern Pipe...,2,1
4,"Oil prices soar to all-time record, posing new...",2,1


Cell 6 — Compare Topics vs Real Labels

In [6]:
import pandas as pd

pd.crosstab(train_df["label"], train_df["topic"])

topic,0,1,2,3
label,,,,
0,514,4363,21592,3531
1,196,374,6734,22696
2,6085,22175,991,749
3,21160,4968,1872,2000


## LDA Topic Modeling Results & Interpretation

### Objective
Apply Latent Dirichlet Allocation (LDA) to discover hidden thematic structure in AG News without using labels. This fulfills the **Core Classical NLP** requirement for unsupervised learning.

### Key Findings
1. **Topic Coherence:** LDA identified 4 distinct topics with clear semantic meaning:
   - **Topic 0 - Sci/Tech:** `technology, computer, internet, software, microsoft`
   - **Topic 1 - Business:** `million, stock, oil, percent, price, reuters`
   - **Topic 2 - World:** `iraq, killed, minister, president, people`
   - **Topic 3 - Sports:** `game, victory, season, win, team`

2. **Label Alignment:** Cross-tabulation shows strong correspondence between unsupervised topics and true news categories, proving latent structure exists in the dataset.

| True Label | Dominant LDA Topic | Docs Aligned |
| --- | --- | --- |
| 0: World | Topic 2 | 21,592 / 30,000 |
| 1: Sports | Topic 3 | 22,696 / 30,000 |
| 2: Business | Topic 1 | 22,175 / 30,000 |
| 3: Sci/Tech | Topic 0 | 21,160 / 30,000 |

3. **Limitation:** Topic 3 mixes sports and political terms like `election`. This is expected in probabilistic models where word co-occurrence drives clustering.

### Conclusion
Despite being unsupervised, LDA successfully recovered human-interpretable news domains. This validates the dataset's semantic quality and completes our multi-paradigm analysis: supervised classification, semantic retrieval, and unsupervised discovery.